In [ ]:
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('load_ext', 'autoreload')
    ipython.run_line_magic('autoreload', '2')
else:
    print("could not load extension")

In [ ]:
import getpass

user = getpass.getuser()
from omni.isaac.kit import SimulationApp

# Set the path below to your desired nucleus server
# Make sure you installed a local nucleus server before this
# and that isaacsim is running
simulation_app = SimulationApp({"livesync_usd": f"omniverse://localhost/Users/{user}/test.usd"})

In [ ]:
import carb
from omni.physx import get_physx_scene_query_interface  # for raycasting e.g raycast_closest()
from omni.isaac.core import World
from omni.isaac.core.objects import DynamicCuboid, DynamicSphere, DynamicCone
from omni.timeline import get_timeline_interface
from omni.isaac.core.utils.stage import get_current_stage
import omni.isaac.core.utils.prims as prims_utils
from omni.usd import get_prim_at_path
import omni.isaac.core.utils.mesh as mesh_utils


import matplotlib.pyplot as plt
from pprint import pprint
import numpy as np
from scipy.interpolate import PchipInterpolator
import typing
from pxr import Usd, UsdGeom, Gf

# Setup Simple Scene


In [ ]:
world = World(stage_units_in_meters=1.0)
stage = get_current_stage()
world.clear()
# world.scene.add_default_ground_plane()
# A render/ step or an update call is needed to reflect the changes to the opened USD in Isaac Sim GUI
#  Note: avoid pressing play/ pause or stop in the GUI in this workflow.
world.render()

# Create the Sun
sun = prims_utils.create_prim(
    "/World/Sun",
    "SphereLight",
    position=[1.0, 1.0, 100.0],
    attributes={
        "inputs:radius": 1,
        "inputs:intensity": 5e3,
        "inputs:color": (1.0, 1.0, 1.0)
    }
)
world.render()

In [ ]:

# IMPORTANT: First time: copy paste the usd from the github repository to the nucleus server
# from: <path to the repository>/jaops-sim/scenes/assets/simplified_surface_solar_panel.usd
# to: omniverse://localhost/Users/<your username>/simplified_surface_solar_panel.usd
# (you can copy paste within the IsaacSim Content Window)


usd_path = f"omniverse://localhost/Users/ubuntu/moon_environnement.usdc"#
prim_path = "/World/Lunar_Base"

# create the prim
prim = prims_utils.create_prim(prim_path=prim_path, usd_path=usd_path)
world.render()

timeline = get_timeline_interface()
timeline.play()
for _ in range(2):
    simulation_app.update()
timeline.pause()

# Classes

In [ ]:
from Lunar_Dust import LunarDust 
from Panel_Class import Panel

# Tests

In [ ]:
# Rover prim introduction

usd_path = f"omniverse://localhost/Users/ubuntu/Rover_e_panneau_orig.usd"#solar_panel_final.usdc"
rover_prim_path = prim_path + "/Rover"

rover = prims_utils.create_prim(prim_path=rover_prim_path, usd_path=usd_path, scale=[20,20,20], position = [20, -40, 20])


world.render()





In [ ]:
# Definition of paths and starting of classes

panel_prim_path = rover_prim_path + "/solo_solar_pannel"

dust_model = LunarDust(world)
dust_model.reduction_altitude = 26.5  ####### for test purposes
dust_model.plot()


panel_path = panel_prim_path + "/cilindro_est_centro/pannello_base"
panel_color_path = panel_prim_path + "/_materials/Material/Principled_BSDF" 
panel = prim.GetPrimAtPath(panel_path)


dust_model.update(0, panel)
dust_model.update_color(panel_color_path)
original_color = dust_model.original_color

In [ ]:
# TEST: manual time passing; the lunar dust will accumulate during time 
# up until the maximum coverage value specified in the class definition

import time

dust_model.reset_accumulation()
dust_model.set_color(panel_color_path, original_color)



i=0
coverage_points = []
time_points = []
while i < 200:
    
    dust_model.update(i*10000000, panel)
    dust_model.update_color(panel_color_path)

    time.sleep(0.8)
    coverage_points.append(dust_model.coverage * 100)
    time_points.append(i*100)
    
    i+=1

    
print(coverage_points)
plt.figure()
plt.plot(time_points, coverage_points)
plt.grid(True)
plt.xlabel("Seconds")
plt.ylabel("Covered surface (%)")
plt.title("Covered surface by time")
plt.show()

## Testing with Simulation Time

In [ ]:
prims_utils.delete_prim(rover_prim_path)
world.render()

usd_path = f"omniverse://localhost/Users/ubuntu/solar_panel_final_turned.usd"
panel_prim_path = prim_path + "/Panel"

panel_structure = prims_utils.create_prim(prim_path=panel_prim_path, usd_path=usd_path, scale=[20,20,20], position = [20, -40, 15])
world.render()

panel = prims_utils.get_prim_at_path(panel_prim_path)

dust_model = LunarDust(world)
dust_model.reduction_altitude = 6.6 ##### FOR NUMERICAL TESTING PURPOSES

panel_path = panel_prim_path + "/cilindro_est_centro/pannello_base"
panel_color_path = panel_prim_path + "/_materials/Material/Principled_BSDF" 
panel = prim.GetPrimAtPath(panel_path)

dust_model.update(0, panel)
dust_model.update_color(panel_color_path)
original_color = dust_model.original_color

In [ ]:
# LUNAR DUST TESTING WITH SIMULATION TIME

import omni.timeline

dust_model.reset_accumulation()
dust_model.reduction_altitude = 6.5 #######
timeline = omni.timeline.get_timeline_interface()
timeline.play()


j=0
last_lunar_dust_call = timeline.get_current_time()
while j<100000:
    current_time = timeline.get_current_time()
    #print(current_time)


    if current_time - last_lunar_dust_call > 5:
        dust_model.update(current_time, panel_structure)
        dust_model.update_color(panel_color_path)
        last_lunar_dust_call = current_time

    current_time = timeline.get_current_time()
    simulation_app.update()

    j+=1

In [ ]:
# TESTING WITH SIMULATION TIME AND PANEL CLASS
# Still not working yet, need to adjust the panel class

import omni.timeline

timeline = omni.timeline.get_timeline_interface()
timeline.play()

import omni.isaac.core.utils.rotations as rotations_utils
dust_model.reduction_altitude = 6.5
panel_prim_path = prim_path + "/Panel/Solar_Generator/Panel"
panel_test = Panel(panel_prim_path)

j=0
last_lunar_dust_call = timeline.get_current_time()
while j<100000:
    current_time = timeline.get_current_time()
    #print(previous_time)


    if current_time - last_lunar_dust_call > 100:
        print("Current time: ", current_time)
        print("")
        print(current_time - last_lunar_dust_call)
        panel_test.update(stage, current_time)
        panel_test.display_state()
        print("----------------------------------------------------------------------------------------")
        last_lunar_dust_call = current_time

    current_time = timeline.get_current_time()
    simulation_app.update()

    j+=1